# Generating RAG Answers and Evaluating

In [17]:
# import ground truth dataset
import pandas as pd

df_ground_truth = pd.read_csv("data/ground_truth-new-v1.csv")
ground_truth = df_ground_truth.to_dict(orient="records")
model = "qwen/qwen3.5-9b"

In [18]:
# filtering to llm zoomcamp docs
from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [19]:
# create look up table for original FAQ documents
doc_idx = {}

for doc in documents:
    doc_idx[doc["id"]] = doc

In [20]:
import os
from dotenv import load_dotenv
load_dotenv(dotenv_path="../../01_module_agentic_rag/.env")
from openai import OpenAI

openai_client = OpenAI(
    api_key=os.getenv("LMSTUDIO_API_KEY"),
    base_url=os.getenv("LMSTUDIO_HOST")
)

In [21]:
from evaluation_utils import LMStudioRAGWithUsage

assistant = LMStudioRAGWithUsage(
    index=index,
    llm_client=openai_client,
)

Note: eval_utils uses the search boosts selected in the search tuning lesson: question=1.0, answer=2.0, and section=0.1. *Although, I received different results with qwen, we stick with theses for now*

In [22]:
# Running the rag with selected question and getting generated answer
rec = ground_truth[0]
question = rec["question"]

answer_llm = assistant.rag(question)
answer_llm

"I don't know."

In [23]:
assistant.total_cost()

AttributeError: 'CompletionUsage' object has no attribute 'input_tokens'

In [ ]:
# Retrieving original answer from FAQ
doc_id = rec["document"]
original_doc = doc_idx[doc_id]
answer_orig = original_doc["answer"]

answer_orig

'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'

In [ ]:
# Save both answers to one record
rag_result = {
    "question": question,
    "answer_llm": answer_llm,
    "answer_orig": answer_orig,
    "document": doc_id,
}

rag_result

{'question': 'Is it okay to join the course late if I just found it now?',
 'answer_llm': 'Yes, you can join the course even if you just discovered it now. However, please note that if your goal is to receive a certificate, you must submit your project while submissions are still being accepted; otherwise, you may not qualify for one.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

In [26]:
# Create function that goes through all questions and generates answer and retrieves original answer (llm-as-judge)
def generate_rag_answer(rec):
    question = rec["question"]
    doc_id = rec["document"]
    original_doc = doc_idx[doc_id]

    answer_llm = assistant.rag(question)
    answer_orig = original_doc["answer"]

    result = {
        "question": question,
        "answer_llm": answer_llm,
        "answer_orig": answer_orig,
        "document": doc_id,
    }

    return result

This calls the LLM once per ground truth question, so it can take some time. We setup the process to run in parallel and track progress.

In [ ]:
# Quick test on one question
answer_record = generate_rag_answer(ground_truth[0])
answer_record

{'question': 'Is it okay to join the course late if I just found it now?',
 'answer_llm': 'Yes, it is okay to join the course late if you just discovered it. According to the provided context, you can start learning and even submit homework while the submission form is open without formally registering. However, please note that if your goal is to receive a certificate, you must submit your capstone project while the course is still accepting submissions.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

In [ ]:
assistant.reset_usage()

In [24]:
# Importing parallel processor helper
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [25]:
# Now run with parallel processing
with ThreadPoolExecutor(max_workers=16) as pool:
    results = map_progress(pool, ground_truth, generate_rag_answer)

  0%|          | 0/583 [00:00<?, ?it/s]

In [27]:
# Collecting answers
answers = []

for answer_record in results:
    answers.append(answer_record)

In [29]:
answers

[{'question': 'I found the course last minute, is it too late to sign up?',
  'answer_llm': "I don't know. The provided context does not contain information regarding whether there is a deadline to sign up for the course or if it is too late to register after finding the course at the last minute.",
  'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'document': '74eb249bbf'},
 {'question': 'Can I still join if I start now, but what about the certificate?',
  'answer_llm': 'Yes, you can still join the course now. Regarding the certificate, you will only receive one if you are able to submit your capstone project while a live cohort is still accepting submissions.\n\nYou can work through the course material in self-paced mode right now, but to get the certificate, you must finish by completing your capstone project and the required peer reviews before the project submission window closes. If you wa

In [ ]:
assistant.total_cost()

In [30]:
df_answers = pd.DataFrame(answers)
df_answers.to_csv("data/rag-answers-new.csv", index=False)

# LLM-As-Judge 

For offline evaluation, we have three things:

    the original FAQ answer
    the question generated from that answer
    the answer generated by our RAG pipeline

An LLM judge is another LLM call that compares these three pieces. We ask it whether the RAG answer recovers the same information as the original answer.

This evaluates the full RAG flow in one pass:

    search: did we retrieve context that contains the answer?
    prompt: did we give the model enough context to answer?
    LLM: did the model produce a useful answer from that context?

If the judge marks an answer as bad, we still need to look at the example. The judge tells us where to investigate. It doesn't replace reading the failing cases.

In [31]:
# loading answers we created
import pandas as pd

df_answers = pd.read_csv("data/rag-answers-new.csv")
answers = df_answers.to_dict(orient="records")

In [32]:
# Creating the output format
from pydantic import BaseModel, Field
from typing import Literal

class AnswerEvaluation(BaseModel):
    reasoning: str = Field(
        description="Reasoning about the quality of the answer."
    )
    score: Literal["good", "bad"] = Field(
        description="'good' if the answer is correct and complete, 'bad' otherwise."
    )

In [33]:
# Creating the judge prompt
aqa_judge_instructions = """
You are an expert evaluator. You will be given:
1. A question from a student
2. The original answer from the FAQ (ground truth)
3. An answer generated by an AI assistant

Your task is to decide if the AI answer is semantically equivalent to
the original answer.

Rules:
- The AI answer does NOT need to be word-for-word identical
- It should convey the same key information
- Extra detail is fine as long as the core answer is correct
- Mark 'bad' only if the AI answer is wrong or misses the key point

Be fair and focus on correctness, not style.
""".strip()

In [34]:
# Create prompt template
aqa_judge_prompt = """
Question:
{question}

Original Answer (ground truth):
{answer_orig}

AI Answer:
{answer_llm}
""".strip()

In [35]:
import os
from dotenv import load_dotenv
load_dotenv(dotenv_path="../../01_module_agentic_rag/.env")
from openai import OpenAI

openai_client = OpenAI(
    api_key=os.getenv("LMSTUDIO_API_KEY"),
    base_url=os.getenv("LMSTUDIO_HOST")
)

In [38]:
rec = answers[0]
rec

{'question': 'I found the course last minute, is it too late to sign up?',
 'answer_llm': "I don't know. The provided context does not contain information regarding whether there is a deadline to sign up for the course or if it is too late to register after finding the course at the last minute.",
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

In [39]:
# Create Judge prompt
prompt = aqa_judge_prompt.format(
    question=rec["question"],
    answer_orig=rec["answer_orig"],
    answer_llm=rec["answer_llm"]
)
prompt

"Question:\nI found the course last minute, is it too late to sign up?\n\nOriginal Answer (ground truth):\nYes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.\n\nAI Answer:\nI don't know. The provided context does not contain information regarding whether there is a deadline to sign up for the course or if it is too late to register after finding the course at the last minute."

In [42]:
from evaluation_utils import llm_structured_retry_chat_completions

In [44]:
# Calling the judge
eval_result, usage = llm_structured_retry_chat_completions(
    openai_client,
    aqa_judge_instructions,
    prompt,
    AnswerEvaluation,
)

eval_result

AnswerEvaluation(reasoning='The AI answer claims that the provided context does not contain the necessary information, but it actually does contain an answer in the original FAQ ("Yes, but..."). The AI incorrectly concludes there is no information when a clear response exists. This makes the AI answer factually wrong compared to the ground truth.', score='bad')

In [ ]:
# Checking cost
calc_price(usage)

In [46]:
# Inputting logic as a callable function
def evaluate_aqa(question, answer_orig, answer_llm, model=model):
    prompt = aqa_judge_prompt.format(
        question=question,
        answer_orig=answer_orig,
        answer_llm=answer_llm
    )

    result, usage = llm_structured_retry_chat_completions(
        openai_client,
        aqa_judge_instructions,
        prompt,
        AnswerEvaluation,
        model=model,
    )

    return result, usage

In [47]:
# Test on first record
eval_result, usage = evaluate_aqa(
    question=rec["question"],
    answer_orig=rec["answer_orig"],
    answer_llm=rec["answer_llm"]
)

eval_result

AnswerEvaluation(reasoning="The AI assistant explicitly states that it cannot answer the question because the provided context lacks specific information about registration deadlines or acceptance of late sign-ups. However, the ground truth (original answer) directly answers 'Yes' and provides a condition for receiving a certificate. By claiming the information is missing when it is actually present in the ground truth, the AI has failed to retrieve or utilize the necessary knowledge to provide the correct response. This constitutes a failure in correctness and semantic equivalence.", score='bad')

In [48]:
# Running eval on all answers
def judge_record(rec):
    eval_result, usage = evaluate_aqa(
        question=rec["question"],
        answer_orig=rec["answer_orig"],
        answer_llm=rec["answer_llm"]
    )

    result = {
        "question": rec["question"],
        "document": rec["document"],
        "score": eval_result.score,
        "reasoning": eval_result.reasoning,
    }

    return result, usage

In [49]:
from concurrent.futures import ThreadPoolExecutor

with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, answers, judge_record)

  0%|          | 0/583 [00:00<?, ?it/s]

In [50]:
# Splitting results
evaluations = []
usages = []

for evaluation, usage in results:
    evaluations.append(evaluation)
    usages.append(usage)

In [51]:
evaluations

[{'question': 'I found the course last minute, is it too late to sign up?',
  'document': '74eb249bbf',
  'score': 'bad',
  'reasoning': "The AI answer claims that the information is missing and says 'I don't know', whereas the original answer explicitly states 'Yes, but...' followed by a condition about receiving a certificate. The ground truth provides a specific instruction/answer (submit your project while still accepting submissions to get a certificate), while the AI fails to provide this key information based on the context it should have processed. Therefore, the AI answer is factually incorrect relative to the ground truth."},
 {'question': 'Can I still join if I start now, but what about the certificate?',
  'document': '74eb249bbf',
  'score': 'good',
  'reasoning': "The AI answer correctly affirms that the user can join now (matching 'Yes, but...'). It accurately conveys the core condition regarding the certificate: it must be submitted while submissions are being accepted.

In [56]:
df_eval = pd.DataFrame(evaluations)
df_eval

,question,document,score,reasoning
0,"I found the course last minute, is it too late...",74eb249bbf,bad,The AI answer claims that the information is m...
1,"Can I still join if I start now, but what abou...",74eb249bbf,good,The AI answer correctly affirms that the user ...
2,Is it possible to enroll even though the deadl...,74eb249bbf,good,The AI answer correctly identifies that enroll...
3,"Hey, I just saw this—am I eligible to join and...",74eb249bbf,good,The AI answer correctly identifies the core co...
4,"Since I discovered the course recently, can I ...",74eb249bbf,good,The AI answer correctly conveys the core messa...
...,...,...,...,...
578,"Hey, why is my EU-region Logfire token throwin...",46efd1088d,good,The AI answer is semantically equivalent to th...
579,I'm getting unauthorized errors—should I manua...,46efd1088d,good,The AI answer correctly identifies that manual...
580,My .env file has the right token but it still ...,46efd1088d,good,The AI answer correctly identifies the primary...
581,"Is my LOGFIRE_TOKEN actually a write token, an...",46efd1088d,good,The AI answer correctly identifies the two mai...


In [ ]:
calc_total_price(usages)

In [57]:
good_count = (df_eval["score"] == "good").sum()
total_count = len(df_eval)
print(f"Good: {good_count}/{total_count} = {good_count/total_count:.2%}")

Good: 506/583 = 86.79%


In [61]:
# Reviewing bad cases
df_eval[df_eval["score"] == "bad"].head()

,question,document,score,reasoning
0,"I found the course last minute, is it too late...",74eb249bbf,bad,The AI answer claims that the information is m...
8,"So basically, registration is just for them to...",977bf7786c,bad,The AI answer confirms the user's premise ('yo...
16,Can I begin whenever I want or do I need to wa...,04919992b3,bad,The AI answer fails to address the core of the...
18,Is the homework format basically the same as t...,04919992b3,bad,The user asks whether homework format is basic...
26,Can I just study at my own pace without joinin...,69d122f12e,bad,The AI answer states there is no direct answer...


In [62]:
df_eval.to_csv("data/rag-evaluations-new.csv", index=False)